# Step 1: Data Loader
Downloads the NYC TLC Yellow Taxi dataset and synthesizes GPS coordinates from zone IDs.

**To run:** Use `Kernel → Restart & Run All` to execute all cells in order.

In [12]:
# Run from project root
import os
if os.path.basename(os.getcwd()) in ['data', 'src', 'models', 'simulation']:
    os.chdir('../..')
print('Working directory:', os.getcwd())

Working directory: /Users/krunal/Desktop/final-project-team-tle


In [ ]:
import os
import requests
import yaml
import pandas as pd

class DataLoader:
    """
    Downloads and preprocesses the NYC TLC Taxi Dataset.
    Handles streaming download, coordinate synthesis from zone IDs,
    and cleaning (GPS anomalies, economic outliers).
    """
    def __init__(self, config_path="config/dev_config.yaml"):
        # Load environment-specific configurations (Dev vs Prod limits)
        with open(config_path, "r") as file:
            self.config = yaml.safe_load(file)
            
        self.data_config = self.config["data"]
        self.raw_dir = self.data_config["raw_dir"]
        self.processed_dir = self.data_config["processed_dir"]
        
        # Ensure immutable raw and processed directories exist
        os.makedirs(self.raw_dir, exist_ok=True)
        os.makedirs(self.processed_dir, exist_ok=True)
        
    def _get_file_name(self):
        """Generates the absolute parquet filename dynamically based on the YAML time config."""
        return f"{self.data_config['taxi_type']}_tripdata_{self.data_config['year']}-{self.data_config['month']:02d}.parquet"
        
    def download_data(self):
        """
        Streams the massive (~2GB) Parquet file directly from AWS Cloudfront.
        We stream in 8KB chunks to prevent RAM overflow locally.
        """
        file_name = self._get_file_name()
        file_url = f"{self.data_config['base_url']}/{file_name}"
        local_path = os.path.join(self.raw_dir, file_name)
        
        # Idempotent execution: Do not download if it already exists locally
        if os.path.exists(local_path):
            print(f"File {file_name} already exists. Skipping download.")
            return local_path
            
        print(f"Downloading {file_url} to {local_path}...")
        response = requests.get(file_url, stream=True)
        if response.status_code == 200:
            with open(local_path, "wb") as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            print("Download complete.")
        else:
            raise Exception(f"Failed to download data. HTTP Status: {response.status_code}")
            
        return local_path

    def clean_data(self, local_path):
        """Load raw parquet, remove invalid trips (bad coords, negative fares, zero distance), save cleaned version."""
        print(f"Loading data from {local_path}...")
        df = pd.read_parquet(local_path)
        
        initial_len = len(df)
        print(f"Initial raw data length: {initial_len}")
        
        # Check if dataset has raw coords or zone IDs
        if 'pickup_latitude' in df.columns:
            lat_col, lon_col = 'pickup_latitude', 'pickup_longitude'
            d_lat_col, d_lon_col = 'dropoff_latitude', 'dropoff_longitude'
        elif 'PULocationID' in df.columns:
            import numpy as np
            import geopandas as gpd
            from zipfile import ZipFile
            import io
            
            print("Notice: NYC TLC retroactively removed exact coordinates from historical Parquet files.")
            print("Synthesizing exact GPS coordinates based on official TLC Zone Centroids and Geodesic Projection...")
            
            # Download and extract the official TLC Taxi Zone shapefile
            zone_url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zones.zip"
            zone_path = os.path.join(self.raw_dir, "taxi_zones")
            if not os.path.exists(zone_path):
                print("Downloading official TLC Taxi Zone Shapefile...")
                os.makedirs(zone_path, exist_ok=True)
                z_response = requests.get(zone_url)
                with ZipFile(io.BytesIO(z_response.content)) as z:
                    z.extractall(zone_path)
                    
            print("Mapping PULocationID to Geographic Bounding Boxes and Centroids...")
            # Handle both flat and nested extraction (zip may extract into a subfolder)
            shp_path = os.path.join(zone_path, "taxi_zones.shp")
            if not os.path.exists(shp_path):
                shp_path = os.path.join(zone_path, "taxi_zones", "taxi_zones.shp")
            gdf = gpd.read_file(shp_path)
            
            # Evaluate bounds for Zonal Bounding Box Distribution (EPSG:4326)
            gdf_4326 = gdf.to_crs("EPSG:4326")
            bounds = gdf_4326.bounds
            gdf['minx'] = bounds['minx']
            gdf['miny'] = bounds['miny']
            gdf['maxx'] = bounds['maxx']
            gdf['maxy'] = bounds['maxy']
            
            # Calculate centroids for the corridor bearing math later
            centroids_latlon = gdf_4326.geometry.centroid
            gdf['centroid_lon'] = centroids_latlon.x
            gdf['centroid_lat'] = centroids_latlon.y
            
            zone_dict = gdf.set_index('LocationID')[['minx', 'miny', 'maxx', 'maxy', 'centroid_lat', 'centroid_lon']].to_dict('index')
            
            # Fallback coordinate
            base_lat, base_lon = 40.7580, -73.9855
            
            # Extract bounds for each trip based on PULocationID
            minx = df['PULocationID'].map(lambda x: zone_dict.get(x, {}).get('minx', base_lon)).fillna(base_lon)
            miny = df['PULocationID'].map(lambda x: zone_dict.get(x, {}).get('miny', base_lat)).fillna(base_lat)
            maxx = df['PULocationID'].map(lambda x: zone_dict.get(x, {}).get('maxx', base_lon)).fillna(base_lon)
            maxy = df['PULocationID'].map(lambda x: zone_dict.get(x, {}).get('maxy', base_lat)).fillna(base_lat)
            
            # Apply Zonal Bounding Box Distribution so pickups are scattered uniformly across their true geographic zone
            np.random.seed(42)
            df['pickup_longitude'] = np.random.uniform(minx, maxx)
            df['pickup_latitude'] = np.random.uniform(miny, maxy)

            # Point-in-polygon refinement: resample any pickups that landed outside
            # their actual zone polygon (e.g., in water or neighboring zones).
            # Critical for irregular coastal zones and nationwide scaling to suburban/rural areas.
            import shapely
            zone_polygons = gdf_4326.set_index('LocationID')['geometry'].to_dict()
            resampled_total = 0

            for zone_id, polygon in zone_polygons.items():
                idx = df.index[df['PULocationID'] == zone_id]
                if len(idx) == 0:
                    continue
                bminx, bminy, bmaxx, bmaxy = polygon.bounds
                for _ in range(10):
                    lons = df.loc[idx, 'pickup_longitude'].values
                    lats = df.loc[idx, 'pickup_latitude'].values
                    outside = ~shapely.contains(polygon, shapely.points(lons, lats))
                    if not outside.any():
                        break
                    n = outside.sum()
                    resampled_total += n
                    df.loc[idx[outside], 'pickup_longitude'] = np.random.uniform(bminx, bmaxx, n)
                    df.loc[idx[outside], 'pickup_latitude'] = np.random.uniform(bminy, bmaxy, n)

            print(f"Point-in-polygon refinement: resampled {resampled_total} points to stay within zone boundaries.")

            print("Computing corridor-aware dropoff bearings from zone centroids...")
            # Use DOLocationID centroids to compute the TRUE commute bearing for each trip.
            # This preserves realistic corridor patterns (e.g., Bronx -> Manhattan in AM).
            # Gaussian noise (s=0.3 rad ~17 degrees) simulates street-level variation.
            
            do_lat = df['DOLocationID'].map(lambda x: zone_dict.get(x, {}).get('centroid_lat', None))
            do_lon = df['DOLocationID'].map(lambda x: zone_dict.get(x, {}).get('centroid_lon', None))
            
            # Forward azimuth (bearing) from pickup centroid to dropoff centroid
            pu_lat_rad = np.radians(df['pickup_latitude'])
            pu_lon_rad = np.radians(df['pickup_longitude'])
            do_lat_rad = np.radians(do_lat)
            do_lon_rad = np.radians(do_lon)
            dlon = do_lon_rad - pu_lon_rad
            
            x = np.sin(dlon) * np.cos(do_lat_rad)
            y = np.cos(pu_lat_rad) * np.sin(do_lat_rad) - np.sin(pu_lat_rad) * np.cos(do_lat_rad) * np.cos(dlon)
            true_bearing = np.arctan2(x, y)  # radians, [-pi, pi]
            
            # Add Gaussian noise to simulate street-level variation around the corridor
            corridor_noise = np.random.normal(0, 0.3, size=len(df))
            angles = true_bearing + corridor_noise
            
            # Fallback: where DOLocationID was unknown, use random angle
            missing_mask = do_lat.isna()
            angles[missing_mask] = np.random.uniform(0, 2 * np.pi, size=missing_mask.sum())
            
            R = 3958.8  # Earth radius in miles
            dist_ratio = df['trip_distance'] / R
            
            lat1 = np.radians(df['pickup_latitude'])
            lon1 = np.radians(df['pickup_longitude'])
            
            lat2 = np.arcsin(np.sin(lat1) * np.cos(dist_ratio) + np.cos(lat1) * np.sin(dist_ratio) * np.cos(angles))
            lon2 = lon1 + np.arctan2(np.sin(angles) * np.sin(dist_ratio) * np.cos(lat1), np.cos(dist_ratio) - np.sin(lat1) * np.sin(lat2))
            
            df['dropoff_latitude'] = np.degrees(lat2)
            df['dropoff_longitude'] = np.degrees(lon2)
            
            lat_col, lon_col = 'pickup_latitude', 'pickup_longitude'
            d_lat_col, d_lon_col = 'dropoff_latitude', 'dropoff_longitude'
        else:
            raise ValueError("Dataset cannot be mapped geographically.")
        
        # 2. Strict Tri-State Area Spatial Bounding Box
        df = df[
            (df[lat_col].between(40.5, 41.2)) & 
            (df[lon_col].between(-74.3, -73.7))
        ]
        
        final_len = len(df)
        print(f"Cleaned data length: {final_len} (Dropped {initial_len - final_len} rows)")
        
        # Output to standardized processed layer
        processed_file = os.path.join(self.processed_dir, f"cleaned_{self._get_file_name()}")
        df.to_parquet(processed_file, index=False)
        print(f"Processed data saved to {processed_file}")
        return processed_file

In [14]:
# Run the data loader
loader = DataLoader()
raw_path = loader.download_data()
loader.clean_data(raw_path)

File yellow_tripdata_2025-08.parquet already exists. Skipping download.
Loading data from data/raw/yellow_tripdata_2025-08.parquet...
Initial raw data length: 3574091
Notice: NYC TLC retroactively removed exact coordinates from historical Parquet files.
Synthesizing exact GPS coordinates based on official TLC Zone Centroids and Geodesic Projection...
Mapping PULocationID to Geographic Bounding Boxes and Centroids...


/var/folders/xc/wspxd6_n7k90gnw05920r_yw0000gn/T/ipykernel_75931/3515278424.py:102: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids_latlon = gdf_4326.geometry.centroid


Point-in-polygon refinement: resampled 4103120 points to stay within zone boundaries.
Computing corridor-aware dropoff bearings from zone centroids...
Cleaned data length: 3574091 (Dropped 0 rows)
Processed data saved to data/processed/cleaned_yellow_tripdata_2025-08.parquet


'data/processed/cleaned_yellow_tripdata_2025-08.parquet'